Builing SCD 2 table


In [0]:
# Databricks notebook source
from pyspark.sql import functions as F
#SCD type 2 table
# COMMAND ----------
# Safe to re-run any number of times — only creates the table if it
# doesn't already exist yet. Same columns as bronze.lead_operations,
# plus valid_from / valid_to / is_current for tracking each version's
# active window.
spark.sql("""
    CREATE TABLE IF NOT EXISTS zaitoon_catalog.silver.lead_history AS
    SELECT *,
           CAST(NULL AS TIMESTAMP) AS valid_from,
           CAST(NULL AS TIMESTAMP) AS valid_to,
           CAST(NULL AS BOOLEAN)   AS is_current
    FROM zaitoon_catalog.bronze.lead_operations
    WHERE 1=0
""")
spark.table("zaitoon_catalog.silver.lead_history").printSchema()

# COMMAND ----------
def upsert_to_lead_history(microBatchDF, batchId: int):
    spark_session = microBatchDF.sparkSession
    microBatchDF.createOrReplaceTempView("lead_history_batch_raw")

    # Order this batch's events per lead, and work out each event's
    # valid_to: the next event for that same lead within this batch,
    # or NULL if it's the newest thing that happened.
    spark_session.sql("""
        CREATE OR REPLACE TEMP VIEW lead_history_batch_ordered AS
        SELECT *,
               LEAD(event_timestamp) OVER (
                   PARTITION BY Id ORDER BY event_timestamp
               ) AS next_event_time
        FROM lead_history_batch_raw
    """)

    # Step 1: close out whatever row is currently "open" for any lead
    # that has at least one new event in this batch.
    spark_session.sql("""
        MERGE INTO zaitoon_catalog.silver.lead_history AS target
        USING (
            SELECT Id, MIN(event_timestamp) AS first_new_event_time
            FROM lead_history_batch_ordered
            GROUP BY Id
        ) AS source
        ON target.Id = source.Id AND target.is_current = true
        WHEN MATCHED THEN UPDATE SET
            target.valid_to = source.first_new_event_time,
            target.is_current = false
    """)

    # Step 2: insert every event in this batch as its own versioned row.
    spark_session.sql("""
        INSERT INTO zaitoon_catalog.silver.lead_history
        SELECT * EXCEPT (next_event_time),
               event_timestamp AS valid_from,
               next_event_time AS valid_to,
               CASE WHEN next_event_time IS NULL AND operation_type != 'DELETE'
                    THEN true ELSE false END AS is_current
        FROM lead_history_batch_ordered
    """)

query = (
    spark.readStream
        .table("zaitoon_catalog.bronze.lead_operations")
        .writeStream
        .foreachBatch(upsert_to_lead_history)
        .option("checkpointLocation", "/Volumes/zaitoon_catalog/bronze/raw_events/_checkpoints/lead_history_v2")
        .trigger(availableNow=True)
        .start()
)
query.awaitTermination()

spark.table("zaitoon_catalog.silver.lead_history").count()